# Proyecto 3: Limpieza y Preprocesamiento de Datos con Pandas

## 🎯 Objetivos
- Implementar un flujo de trabajo profesional de limpieza de datos.
- Identificar y cuantificar datos faltantes (*Missing Data*).
- Aplicar estrategias de imputación (Moda) y eliminación de registros.
- Realizar ingeniería de características básica (*Feature Engineering*) extrayendo valores numéricos de texto.
- Detectar valores atípicos (*Outliers*) mediante análisis de distribución.

## 1. Introducción: El Principio GIGO

En ciencia de datos existe un concepto fundamental llamado **GIGO** (*Garbage In, Garbage Out*), que significa "Basura entra, Basura sale". 

Si alimentamos un modelo de análisis o de Machine Learning con datos incompletos, inconsistentes o erróneos, los resultados serán igualmente erróneos, sin importar qué tan avanzado sea el algoritmo. Por ello, la limpieza de datos es la etapa que más tiempo consume (y la más importante) de cualquier proyecto.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('netflix_titles.csv')
df_netflix = pd.read_csv(DATA_PATH)
df_netflix.head()

## 2. Exploración Inicial

Antes de limpiar, debemos entender la "anatomía" de nuestro dataset: ¿Cuántos datos hay? ¿De qué tipo son?

In [ ]:
# Dimensiones del dataset
print(f"Dimensiones: {df_netflix.shape}")

# Tipos de datos por columna
df_netflix.dtypes

## 3. Identificación de Datos Faltantes

Los valores `NaN` (*Not a Number*) son huecos en nuestra información que pueden romper los cálculos matemáticos.

In [ ]:
# Conteo absoluto de valores nulos
missing_count = df_netflix.isnull().sum().sort_values(ascending=False)
print("Valores faltantes por columna:\n", missing_count)

# Cálculo porcentual para priorizar la limpieza
print("\nPorcentaje de datos faltantes:")
for col in df_netflix.columns:
    pct = df_netflix[col].isnull().mean() * 100
    print(f"{col}: {pct:.2f}%")

## 4. El Puente Pedagógico: Estrategias de Resolución

### ¿Eliminar o Imputar?
Cuando encontramos un valor faltante, tenemos dos caminos principales:

1. **Eliminación (`dropna`)**: Útil cuando el porcentaje de datos faltantes es muy bajo (<5%) o cuando la columna es irrelevante para el objetivo del análisis.
2. **Imputación (`fillna`)**: Consiste en rellenar el hueco con un valor estimado.
   - **Datos Numéricos**: Se suele usar la *Media* (promedio) o la *Mediana* (valor central).
   - **Datos Categóricos**: Se usa la *Moda* (el valor que más se repite).

#### Pipeline de Limpieza de Datos
```
  [ Raw Data ] --> [ Exploración ] --> [ Identificación ] --> [ Resolución ] --> [ Clean Data ]
                        |                    |                      |
                    dtypes/shape        isnull()/mean()        Drop / Impute
                                                                 (Mean/Median/Mode)
```

In [ ]:
# Caso A: Eliminar filas donde el director es desconocido
df_clean_dir = df_netflix.dropna(subset=['director']).copy()

# Caso B: Imputar la columna 'rating' usando la Moda
most_common_rating = df_netflix['rating'].mode()[0]
df_netflix['rating'] = df_netflix['rating'].fillna(most_common_rating)

print(f"Se imputó el rating faltante con la moda: {most_common_rating}")

## 5. Ingeniería de Características (*Feature Engineering*)

A menudo, los datos vienen en formatos "humanos" (ej: "90 min") pero necesitamos formatos "computacionales" (ej: `90`).

In [ ]:
# Filtramos solo las películas para extraer la duración en minutos
df_movies = df_netflix[df_netflix['type'] == 'Movie'].copy()

# Extraemos el número usando .str.split() y convertimos a entero
df_movies['duration_min'] = df_movies['duration'].str.split(' ').str[0].astype(int)

df_movies[['title', 'duration', 'duration_min']].head()

## 6. Detección de Valores Atípicos (*Outliers*)

Un valor atípico es una observación que se aleja drásticamente del resto. Pueden ser errores de entrada o casos excepcionales reales.

In [ ]:
import matplotlib.pyplot as plt

# Visualización de la distribución de minutos
df_movies['duration_min'].plot(kind='hist', bins=20, color='teal', edgecolor='black', figsize=(10, 6))
plt.title('Distribución de la Duración de Películas')
plt.xlabel('Minutos')
plt.ylabel('Frecuencia')
plt.show()

# Análisis numérico de los rangos
print("Distribución por rangos:\n", df_movies['duration_min'].value_counts(bins=10).sort_index())

## 📝 Ejercicios

1. **Imputación Avanzada**: Crea una nueva columna llamada `country_filled` donde los valores nulos de `country` sean reemplazados por la cadena `'Unknown'`.
2. **Análisis de Outliers**: Basado en el histograma, ¿cuál es el límite superior de duración que considerarías un *outlier*? Filtra el DataFrame para mostrar solo las películas que superen ese límite.
3. **Limpieza de Cast**: Utiliza `.str.split(',')` para contar cuántos actores tiene cada película y guarda este resultado en una nueva columna llamada `cast_count`.

## 📋 Resumen de Limpieza

| Paso | Herramienta | Propósito |
|---|---|---|
| **Detección** | `.isnull().sum()` | Localizar huecos de información |
| **Eliminación** | `.dropna()` | Quitar registros incompletos irreversibles |
| **Imputación** | `.fillna()` | Rellenar huecos con Media/Mediana/Moda |
| **Transformación** | `.str.split()` | Convertir texto sucio en datos numéricos |
| **Validación** | `.plot(kind='hist')` | Detectar anomalías o errores de entrada |